# dataset

生成训练集
第一版(不支持并发，太慢了):

In [ ]:
import json
import random
from openai import OpenAI
from tqdm import tqdm

INPUT_FILE = "./data/corpus_with_docids.jsonl"
OUTPUT_FILE = "./data/zerogr_train_mini.jsonl"
API_KEY = "sk-372db93fdb7a45aa8b5adefe01ed92e2"
BASE_URL = "https://api.deepseek.com"  

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

# ================= 1. 定义指令模板 (参考 ZeroGR Task Schema) =================
# [cite: 728-772]
INSTRUCTION_TEMPLATES = [
    {
        "intent": "fact",
        "instruction": "Given a question, retrieve factual documents that answer it."
    },
    # {
    #     "intent": "legal",
    #     "instruction": "Retrieve the relevant legal articles or policy clauses corresponding to the query."
    # },
    {
        "intent": "timeline",
        "instruction": "Find documents that describe the latest timeline or updated status of the event."
    },
    {
        "intent": "conflict",
        "instruction": "Find documents that might contain conflicting evidence or refuting details regarding the query."
    }
]

# ================= 2. Prompt 设计 =================
# 这个 Prompt 的作用是让 LLM 扮演用户，根据文档和任务生成问题
QUERY_GEN_PROMPT = """
You are a synthetic data generator.
I will provide you with a document excerpt and a specific retrieval task instruction.
Your goal is to generate a specific User Query (Search Query) that this document would answer perfectly, under the context of the given instruction.

Document Content:
{doc_content}

Task Instruction:
{instruction}

Requirements:
1. The query must be answerable using the provided document.
2. The query should match the style of the "Task Instruction" (e.g., if asking for legal articles, the query should sound like a legal search).
3. Output ONLY the query text, no explanation.
4. Language: Keep the query in the same language as the document (likely Chinese based on your thesis).

User Query:
"""

# ================= 3. 生成函数 =================
def generate_pseudo_query(doc_content, instruction):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat", # 推荐用便宜且强的模型
            messages=[
                {"role": "user", "content": QUERY_GEN_PROMPT.format(
                    doc_content=doc_content, 
                    instruction=instruction
                )}
            ],
            temperature=0.7, # 增加一点创造性，让问题多样化
            max_tokens=60
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error: {e}")
        return None

# ================= 主流程 =================
if __name__ == "__main__":
    results = []
    
    # 读取上一步生成的带DocID的数据
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        corpus = [json.loads(line) for line in f]
    
    print(f"加载了 {len(corpus)} 条文档片段。开始生成指令化查询对...")
    
    # 遍历每个文档片段
    for doc in tqdm(corpus):
        # 策略：每个文档生成 2-3 个不同的指令-查询对
        # 这样可以增加数据的多样性 (Scaling Instruction Fine-tuning [cite: 22])
        
        # 随机选 2 个适合该文档的指令（你也可以根据文档元数据category来指定）
        # 这里为了简单，随机抽取
        selected_templates = random.sample(INSTRUCTION_TEMPLATES, 2)
        
        for temp in selected_templates:
            query = generate_pseudo_query(doc['content'], temp['instruction'])
            
            if query:
                training_sample = {
                    "doc_id": doc['semantic_docid'],  # 目标：语义ID
                    "instruction": temp['instruction'], # 输入1：指令
                    "intent": temp['intent'],         # 元数据：意图
                    "query": query,                   # 输入2：伪查询
                    "source_content": doc['content']  # 保留原文用于人工检查
                }
                results.append(training_sample)

    # 保存
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in results:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            
    print(f"生成完毕！共获得 {len(results)} 条训练数据。")
    print(f"数据已保存至 {OUTPUT_FILE}")

instruction 模板 + prompt 模板

In [ ]:
INSTRUCTION_TEMPLATES = [
    {"intent": "fact", "instruction": "Given a question, retrieve factual documents that answer it."},
    {"intent": "timeline", "instruction": "Find documents that describe the latest timeline or updated status of the event."},
    {"intent": "conflict", "instruction": "Find documents that might contain conflicting evidence or refuting details regarding the query."}
]

QUERY_GEN_PROMPT = """
You are a synthetic data generator.
I will provide you with a document excerpt and a specific retrieval task instruction.
Your goal is to generate a specific User Query (Search Query) that this document would answer perfectly, under the context of the given instruction.

Document Content:
{doc_content}

Task Instruction:
{instruction}

Requirements:
1. The query must be answerable using the provided document.
2. The query should match the style of the "Task Instruction".
3. Output ONLY the query text, no explanation.
4. Language: Keep the query in the same language as the document.

User Query:
"""


## 查看评估结果

定义输入，输出文件

In [1]:
input = "../data/eval_results.jsonl"
output_json = "results_pass.json"
output_jsonl = "../data/zerogr_sft.jsonl"

python脚本：jsonl to json

In [2]:
import json

def process_jsonl_to_json(input_file, output1, output2):
    data_list = []
    
    # 1. 读取 JSONL 文件
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            for line in f:
                # 去掉行尾空格和换行符
                line = line.strip()
                if line:  # 确保行不为空
                    # 将 JSON 字符串解析为 Python 字典
                    record = json.loads(line)
                    data_list.append(record)
        
        # final_training_data = [item for item in data_list if item['eval']['score'] >= 4]
        final_training_data = [item for item in data_list if item['eval']['pass'] == True]

        # 2. 写入 JSON 文件
        with open(output1, 'w', encoding='utf-8') as f:
            # indent=4 让输出的 JSON 文件具备可读性（缩进）
            # ensure_ascii=False 确保中文字符能正常显示
            json.dump(final_training_data, f, indent=4, ensure_ascii=False)
            
        # 3. 写入 JSONL 文件
        with open(output2, 'w', encoding='utf-8') as f:
            for item in final_training_data:
                # 将字典转为 JSON 字符串并添加换行符
                # ensure_ascii=False 确保中文正常
                json_record = json.dumps(item, ensure_ascii=False)
                f.write(json_record + '\n')

        print(f"共{len(final_training_data)}个训练集样本")
        print(f"处理完成！已保存至: {output2}")
        
    except FileNotFoundError:
        print("错误：未找到输入文件。")
    except json.JSONDecodeError as e:
        print(f"错误：JSON 解析失败 - {e}")

# 调用示例
# process_jsonl_to_json('data.jsonl', 'result.json')

# 快速提取高分数据
# final_training_data = [item for item in results if item['eval']['score'] >= 4]

In [4]:
process_jsonl_to_json(input, output_json, output_jsonl)

共599个训练集样本
处理完成！已保存至: ../data/zerogr_sft.jsonl


## Lora微调


## Alpaca格式

In [11]:
import json

INPUT_FILE = "../data/zerogr_sft.jsonl"
OUTPUT_FILE = "/home/aizoo/data/workspace/LLaMA-Factory/data/zerogr_sft.json" # 直接存到 LLaMA-Factory 的 data 目录下

data = []
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        entry = {
            "instruction": row['instruction'], # 比如 "Find documents that..."
            "input": row['query'],             # 比如 "梯度下降最新..."
            "output": row['doc_id']            # 比如 "machine learning optimization..."
        }
        data.append(entry)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"转换完成！已保存到 {OUTPUT_FILE}")

转换完成！已保存到 /home/aizoo/data/workspace/LLaMA-Factory/data/zerogr_sft.json
